# USDA AMS Market News (MARS API) — Local Cash & Terminal Prices

**What it is.** USDA's market reporters publish daily prices/volumes across **grain, livestock,
dairy, poultry, fruit & vegetable** markets. The **MARS API** serves those reports as data.

| | |
|---|---|
| **Coverage** | United States — hundreds of local/terminal markets |
| **Cost / key** | **Free · API key required** (register; higher rate limits) |
| **Sign up** | https://mymarketnews.ams.usda.gov → profile → API key → `.env` `AMS_API_KEY` |
| **Auth** | HTTP Basic — **API key as username**, blank password |
| **Docs** | https://mymarketnews.ams.usda.gov/mars-api |

**Why it's in Tracera.** This is **local cash price** — what elevators actually pay. Combined
with **futures** (`cme.ipynb`) it gives **basis**, the number that drives real marketing
decisions.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Setup — key guard (register free; 403 without a key)

In [2]:
AMS_KEY = get_key("AMS_API_KEY", required=False)
print("AMS_API_KEY present:", bool(AMS_KEY))
if not AMS_KEY:
    print("→ Register free at https://mymarketnews.ams.usda.gov to get an API key, add as AMS_API_KEY.")

AMS_API_KEY present: True


### 2. Core query — list reports, then pull one (auth = key as username)

In [3]:
MARS = "https://marsapi.ams.usda.gov/services/v1.2"

def mars(path, **params):
    """MARS API uses HTTP Basic with the API key as the username."""
    r = requests.get(f"{MARS}/{path}", auth=(AMS_KEY, ""), params=params, timeout=45)
    r.raise_for_status()
    return r.json()

if AMS_KEY:
    reports = pd.DataFrame(mars("reports"))
    print(f"{len(reports)} report definitions available. Search for grain reports:")
    grain = reports[reports["report_title"].str.contains("Corn|Grain", case=False, na=False)]
    display(grain[["slug_id", "report_title"]].head(10))
    # Then fetch one report's data, e.g.:  data = pd.DataFrame(mars(f"reports/{slug_id}"))
else:
    print("Skipped live call — add AMS_API_KEY. The request code above is ready to run.")

1049 report definitions available. Search for grain reports:


,slug_id,report_title
694,2711,Texas Daily Grain Bids
696,2714,Maryland Grain Bids
715,2771,Montana Daily Elevator Grain Bids
727,2787,South Carolina Daily Grain Bids
753,2850,Iowa Daily Cash Grain Bids
754,2851,Ohio Daily Grain Bids
762,2886,Kansas Daily Grain Bids
765,2892,Kentucky Daily Grain Bids
776,2912,Colorado Daily Grain Bids
779,2920,National Weekly Non-GMO/GE Grain Report


### Notes & how to extend
- **Two steps:** `reports` lists every report with a `slug_id`; `reports/{slug_id}` returns that
  report's rows. Add `?q=...` filters (commodity, date) on the detail endpoint.
- **Coverage:** grain bids, livestock auctions, dairy, organic, national/terminal summaries.
- **Basis workflow:** join a local **cash grain** report here to the **futures** close in
  `cme.ipynb` on date → basis (cash − futures).
- A sample key exists in the docs for testing but is being retired — register for your own.